# Lekce 04 - Návrhový vzor Použití nástroje

V této lekci se naučíte návrhový vzor **Použití nástroje** pro AI agenty pomocí Microsoft Agent Framework (Python). Pokryjeme:

- Definování funkčních nástrojů s dekorátorem `@tool` a typovanými parametry
- Poskytování schémat nástrojů, aby model věděl, co každý nástroj dělá
- Řízení spuštění nástroje pomocí `approval_mode`
- Vrácení **strukturovaného výstupu** pomocí Pydantic modelů a `response_format`

Scénář je **agent pro rezervaci cestování**, který může vyhledávat destinace, kontrolovat dostupnost a získávat informace o letech.


## Nastavení


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Definování nástrojů s dekorátorem @tool

Dekorátor `@tool` promění obyčejnou Python funkci na nástroj, který může agent volat.
Klíčové body:

- **Dokumentační řetězec** se stává popisem nástroje, které model vidí.
- **Typové anotace** (včetně `Annotated` s popisy) definují schéma nástroje.
- `approval_mode` určuje, zda uživatel musí schválit každé volání před jeho provedením.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Vytvoření agenta s více nástroji

Předáte všechny tři nástroje klientovi, aby model mohl použít kterékoli z nich, které potřebuje k zodpovězení uživatelovy otázky.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturovaný výstup s nástroji

Nastavením `response_format` na model Pydantic je agent donucen vrátit dobře typovaný JSON objekt místo volného textu. To je užitečné, když následný kód potřebuje výsledek zpracovat programově.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Vzory schvalování nástrojů

Parametr `approval_mode` u `@tool` řídí, zda volání nástroje vyžaduje před spuštěním lidské schválení:

| Režim | Chování |
|---|---|
| `"never_require"` | Nástroj běží automaticky — není potřeba potvrzení uživatele. |
| `"always_require"` | Každé volání musí být uživatelem schváleno před vykonáním. |

Použijte `"always_require"` pro nástroje, které mají vedlejší efekty (např. rezervace letu, stržení peněz z kreditní karty), aby člověk zůstal v procesu.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Shrnutí

V této lekci jste se naučili, jak:

1. **Definovat nástroje** pomocí dekorátoru `@tool` s typovanými parametry a dokumentačními řetězci, které slouží jako schéma nástroje.
2. **Skládat více nástrojů** tak, aby je agent mohl volat postupně a odpovídat na složité dotazy.
3. **Vrátit strukturovaný výstup** předáním modelu Pydantic jako `response_format`.
4. **Řídit schvalování nástrojů** pomocí `approval_mode` pro udržení člověka v procesu u citlivých operací.

Tyto vzory tvoří základ pro vytváření spolehlivých agentů připravených pro produkční použití, kteří se mohou bezpečně připojovat k externím systémům.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Prohlášení o vyloučení odpovědnosti**:
Tento dokument byl přeložen pomocí AI překladatelské služby [Co-op Translator](https://github.com/Azure/co-op-translator). I když usilujeme o přesnost, mějte prosím na paměti, že automatické překlady mohou obsahovat chyby nebo nepřesnosti. Původní dokument v jeho mateřském jazyce by měl být považován za autoritativní zdroj. Pro důležité informace se doporučuje profesionální lidský překlad. Nejsme odpovědni za žádné nedorozumění nebo mylné výklady vyplývající z použití tohoto překladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
